# CMIP6 core fix-plan example

This notebook uses a tiny synthetic CMIP6 dataset and a JSON fix plan to show the public Woodpecker plan API: check a dataset with a plan, dry-run the plan, apply it in memory, and re-check the result.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np

import woodpecker
from woodpecker.testing import make_cmip6

Create a deterministic CMIP6-like dataset where near-surface air temperature is stored in Celsius instead of Kelvin.

In [ ]:
dataset = make_cmip6(overrides={"units": "degC"}, seed=7)
original_values = dataset["tas"].values.copy()

dataset

Write a small fix-plan document. The plan matches CMIP6-like datasets and selects the built-in core units fix.

In [ ]:
workspace = TemporaryDirectory()
plan_path = Path(workspace.name) / "cmip6-core-plan.json"
plan_path.write_text(
    json.dumps(
        {
            "plans": [
                {
                    "id": "cmip6.core_units",
                    "match": {"attrs": {"project_id": "CMIP6"}},
                    "steps": [
                        {"id": "woodpecker.normalize_tas_units_to_kelvin"},
                    ],
                }
            ]
        }
    ),
    encoding="utf-8",
)

plan_path

In [ ]:
findings = woodpecker.check_plan(plan_path, inputs=dataset)
findings.fix_ids

A dry run reports what would change but leaves the dataset untouched.

In [ ]:
dry_run = woodpecker.fix_plan(plan_path, inputs=dataset, write=False)

dry_run.stats, dataset["tas"].attrs["units"], np.allclose(dataset["tas"].values, original_values)

Apply the same plan in memory with `write=True`.

In [ ]:
write = woodpecker.fix_plan(plan_path, inputs=dataset, write=True)

(
    write.stats,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values + 273.15),
)

In [ ]:
recheck = woodpecker.check_plan(plan_path, inputs=dataset)
recheck.has_findings

In [ ]:
workspace.cleanup()